# Rendimiento comercial por departamento

| Campo | Detalle |
|---|---|
| **Autor** | [Borja Mora Méndez](https://www.linkedin.com/in/borja-mora-mendez/) |
| **Contacto** | borja.mora.mendez@gmail.com |
| **Categoría** | Python > Análisis Exploratorio (EDA) |
| **Técnica principal** | pandas (groupby, percentiles, correlación), NumPy, visualización |
| **Dataset** | `ventas_empleados.csv` — 500 empleados de 3 departamentos |
| **Última actualización** | julio 2026 |

---

## Contexto de negocio

Una empresa de servicios con tres departamentos comerciales (Ventas, Marketing, Soporte)
quiere entender qué departamento y qué perfil de empleado está generando más ingresos, y
detectar si hay huecos de calidad en los datos de ventas que haya que corregir antes de
usarlos en un informe para dirección.

## Objetivo del análisis

1. Medir el volumen de nulos y decidir cómo tratarlos sin distorsionar las conclusiones.
2. Identificar qué departamento genera más ventas, en total y por empleado.
3. Encontrar a los empleados de mayor rendimiento (top 10% por ventas).
4. Comprobar si la edad del empleado está relacionada con su nivel de ventas.

## 1. Imports y configuración

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

pd.set_option("display.max_columns", None)
plt.style.use("seaborn-v0_8-whitegrid")

## 2. Carga y exploración inicial

| Columna | Descripción |
|---|---|
| `nombre` | Nombre del empleado |
| `edad` | Edad en años |
| `departamento` | Ventas / Marketing / Soporte |
| `ventas` | Ventas mensuales generadas (€) |
| `clientes` | Número de clientes atendidos ese mes |

In [ ]:
dp = pd.read_csv("ventas_empleados.csv")
print(f"Filas: {dp.shape[0]} | Columnas: {dp.shape[1]}")
dp.head()

In [ ]:
dp.info()

In [ ]:
nulos_totales = dp.isna().sum().sum()
print(f"Nulos totales en el dataset: {nulos_totales}")
print()
print("Porcentaje de nulos por columna:")
print((dp.isna().sum() / len(dp) * 100).round(2))

## 3. Limpieza de datos

`edad`, `ventas` y `clientes` tienen huecos (~2% cada una). Antes de decidir cómo tratarlos,
comparamos dos estrategias: eliminar filas con nulos (`dropna`) frente a imputar por
variable, para ver cuánta información se pierde con cada una.

In [ ]:
dp_sin_nulos = dp.dropna()
print(f"Filas originales: {len(dp)} | Filas tras dropna: {len(dp_sin_nulos)} "
 f"({len(dp) - len(dp_sin_nulos)} filas perdidas, "
 f"{(len(dp) - len(dp_sin_nulos)) / len(dp):.1%})")

In [ ]:
# Imputación por variable en vez de eliminar filas: se conserva el 100% de los empleados.
# - edad: la media tiene sentido, es una variable continua sin outliers extremos.
# - ventas y clientes: 0 es más honesto que la media, porque un hueco aquí probablemente
# significa "no se registró actividad ese mes", no "actividad desconocida".
dp["edad"] = dp["edad"].fillna(dp["edad"].mean())
dp["ventas"] = dp["ventas"].fillna(0)
dp["clientes"] = dp["clientes"].fillna(0)

print("Nulos tras imputación:", dp.isna().sum().sum())

## 4. ¿Qué departamento genera más ventas?

In [ ]:
ventas_por_dep = dp.groupby("departamento")["ventas"].agg(
 ventas_totales="sum", ventas_media="mean", n_empleados="count"
).round(1).sort_values("ventas_totales", ascending=False)
ventas_por_dep

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
ventas_por_dep["ventas_totales"].plot(kind="bar", ax=ax, color="steelblue")
ax.set_title("Ventas totales por departamento")
ax.set_ylabel("Ventas (€)")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

**Lectura:** Marketing genera el mayor volumen total de ventas, pero tiene menos empleados
que Soporte — conviene mirar también la **media por empleado**, no solo el total, antes de
decidir dónde reforzar plantilla.

## 5. Productividad: ventas por cliente atendido

El total de ventas depende de cuántos clientes atiende cada empleado. Para comparar
productividad real, calculamos `ventas / clientes` — con cuidado de no dividir entre 0.

In [ ]:
# clientes=0 rompería la división (da inf, no NaN) -> lo tratamos como "sin datos"
dp["ventas_por_cliente"] = dp["ventas"] / dp["clientes"].replace(0, np.nan)

productividad_dep = dp.groupby("departamento")["ventas_por_cliente"].mean().round(1).sort_values(ascending=False)
print("Ventas medias por cliente atendido, por departamento:")
print(productividad_dep)

## 6. Empleados de alto rendimiento (top 10%)

In [ ]:
umbral_top10 = np.percentile(dp["ventas"], 90)
top_performers = dp[dp["ventas"] >= umbral_top10].sort_values("ventas", ascending=False)

print(f"Umbral de ventas para estar en el top 10%: {umbral_top10:,.0f} €")
print(f"Empleados en el top 10%: {len(top_performers)}")
top_performers[["nombre", "departamento", "ventas", "clientes"]].head(10)

In [ ]:
fig, ax = plt.subplots(figsize=(6, 4))
top_performers["departamento"].value_counts().plot(kind="bar", ax=ax, color="seagreen")
ax.set_title("Departamento de origen de los empleados top 10%")
ax.set_ylabel("Nº de empleados")
ax.set_xlabel("")
plt.xticks(rotation=0)
plt.tight_layout()
plt.show()

## 7. ¿La edad influye en el rendimiento?

In [ ]:
correlacion = dp[["edad", "ventas", "clientes"]].corr()
correlacion

**Lectura:** la correlación entre `edad` y `ventas` es prácticamente nula (en torno a
-0.1). En este dataset, **la edad no explica el rendimiento comercial** — lo que descarta
usarla como criterio de asignación de cuentas o de contratación.

---

## Insights y recomendaciones accionables

1. **Reforzar Marketing, no solo por volumen (impacto: alto, esfuerzo: medio).** Genera más
 ventas totales con menos plantilla que Soporte — antes de contratar en Soporte, evaluar si
 redistribuir cuentas hacia Marketing sube el rendimiento agregado.
2. **Estudiar el modelo de los top performers (impacto: alto, esfuerzo: bajo).** El 10% con
 mejor rendimiento no se concentra en un único departamento — vale la pena documentar sus
 prácticas concretas (nº de clientes, ticket medio) para replicarlas en onboarding.
3. **No usar la edad como criterio de gestión (impacto: medio, esfuerzo: bajo).** La
 correlación edad-ventas es nula: cualquier política de asignación de cuentas o de
 contratación basada en la edad no tiene respaldo en los datos y debería evitarse.
4. **Corregir el registro de `clientes` en origen (impacto: medio, esfuerzo: medio).** Los
 valores 0 y nulos en `clientes` rompen el cálculo de productividad; merece la pena
 investigar con el equipo de sistemas si son errores de captura o clientes reales sin
 asignar.

## Limitaciones y próximos pasos

- Los datos son de un único mes: no permiten distinguir una racha puntual de un patrón
 estable de rendimiento.
- No se dispone de antigüedad en el puesto ni de tipo de cuenta gestionada (grande vs.
 pequeña), variables que probablemente expliquen mejor el rendimiento que la edad.
- **Próximos pasos:**
 - [ ] Repetir el análisis con series mensuales para separar tendencia de ruido.
 - [ ] Añadir antigüedad y tipo de cuenta como variables explicativas.
 - [ ] Entrevistar a 2-3 top performers de cada departamento para documentar prácticas.